# Operator Domain Fine-tuning — BiLSTM-CRF NER

Fine-tunes the multilingual `merged_best.pt` checkpoint on synthetic operator domain data.  
Teaches the model to recognize **tariff names**, **USSD codes**, **service names**, and **operator commands** as `MISC` entities.

**GPU: T4 × 2 recommended**

In [ ]:
!git clone https://github.com/uzbtrust/Uzbek-Operator-NER-From-Scratch.git
%cd Uzbek-Operator-NER-From-Scratch
!pip install -q seqeval pyyaml

## 1. Kaggle Input Paths

In [ ]:
import os
import sys
import json
import shutil
import time
import yaml
from pathlib import Path

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")

MERGED_CKPT   = "/kaggle/input/models/uzbtrust/uzbek-operator-ner/pytorch/uzbek-operator-name/1/merged_best.pt"
EN_CKPT       = "/kaggle/input/models/uzbtrust/en-coll-best/pytorch/en_coll_best/1/en_conll_best.pt"
RU_CKPT       = "/kaggle/input/models/uzbtrust/ru-wikiann-best/pytorch/ru_wikiann_best/1/ru_wikiann_best.pt"

WORD_VOCAB    = "/kaggle/input/datasets/uzbtrust/vocab-json/word_vocab.json"
CHAR_VOCAB    = "/kaggle/input/datasets/uzbtrust/vocab-json/char_vocab.json"
TAG_MAP       = "/kaggle/input/datasets/uzbtrust/tag-map/tag_map.json"

WORKING_DIR   = Path("/kaggle/working")
VOCAB_DIR     = WORKING_DIR / "vocab"
VOCAB_DIR.mkdir(exist_ok=True)

shutil.copy(WORD_VOCAB, VOCAB_DIR / "word_vocab.json")
shutil.copy(CHAR_VOCAB, VOCAB_DIR / "char_vocab.json")
shutil.copy(TAG_MAP,    VOCAB_DIR / "tag_map.json")

print("Vocab files copied to", VOCAB_DIR)
print("Merged checkpoint:", Path(MERGED_CKPT).exists())

## 2. Generate Synthetic Operator Domain Data

In [ ]:
import sys
sys.argv = [
    "",
    "--output_dir", "/kaggle/working/data/operator_domain",
    "--n_per_template", "80",
    "--seed", "42",
]

from data.generate_synthetic import main as gen_synthetic
gen_synthetic()

In [ ]:
domain_dir = Path("/kaggle/working/data/operator_domain")

for split in ["train", "validation", "test"]:
    with open(domain_dir / f"{split}.json") as f:
        data = json.load(f)
    misc_count = sum(1 for s in data for t in s["tags"] if "MISC" in t)
    print(f"{split:12s}: {len(data):4d} samples | {misc_count:4d} MISC tokens")

print("\nSample sentences:")
with open(domain_dir / "train.json") as f:
    samples = json.load(f)

for s in samples[:5]:
    pairs = list(zip(s["tokens"], s["tags"]))
    highlighted = " ".join(f"[{w}|{t}]" if t != "O" else w for w, t in pairs)
    print(f"  [{s['lang']}] {highlighted}")

## 3. Load Vocab & Build Model

In [ ]:
from data.vocab import Vocabulary, CharVocabulary, TagMap
from model.ner_model import BiLSTMCRF

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

word_vocab = Vocabulary.load(VOCAB_DIR / "word_vocab.json")
char_vocab = CharVocabulary.load(VOCAB_DIR / "char_vocab.json")
tag_map    = TagMap.load(VOCAB_DIR / "tag_map.json")

print(f"Words: {len(word_vocab):,}  |  Chars: {len(char_vocab)}  |  Tags: {len(tag_map)}")
print(f"Tags: {list(tag_map.tag2idx.keys())}")

In [ ]:
model = BiLSTMCRF(
    vocab_size   = len(word_vocab),
    num_chars    = len(char_vocab),
    num_tags     = len(tag_map),
    word_dim     = cfg["embeddings"]["word_dim"],
    char_dim     = cfg["embeddings"]["char_dim"],
    char_filters = cfg["embeddings"]["char_filters"],
    char_kernel  = cfg["embeddings"]["char_kernel_size"],
    num_langs    = cfg["model"]["num_langs"],
    lang_dim     = cfg["embeddings"]["lang_dim"],
    hidden_size  = cfg["model"]["hidden_size"],
    num_layers   = cfg["model"]["num_layers"],
    dropout      = cfg["model"]["dropout"],
)

ckpt = torch.load(MERGED_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded from merged checkpoint")
print(f"Parameters: {total:,} total | {trainable:,} trainable")

## 4. Freeze Word Embeddings (optional)

Freezing word embeddings keeps FastText representations intact and speeds up training.

In [ ]:
FREEZE_WORD_EMB = True

if FREEZE_WORD_EMB:
    for param in model.embedding.word_embedding.parameters():
        param.requires_grad = False
    print("Word embeddings frozen")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 5. Prepare Domain Datasets

In [ ]:
from torch.utils.data import DataLoader
from data.preprocess import NERDataset, collate_batch, load_raw_data

MAX_SEQ  = cfg["data"]["max_seq_len"]
MAX_WORD = cfg["data"]["max_word_len"]
BS       = 32

train_samples = load_raw_data(domain_dir / "train.json")
val_samples   = load_raw_data(domain_dir / "validation.json")
test_samples  = load_raw_data(domain_dir / "test.json")

train_ds = NERDataset(train_samples, word_vocab, char_vocab, tag_map, MAX_SEQ, MAX_WORD)
val_ds   = NERDataset(val_samples,   word_vocab, char_vocab, tag_map, MAX_SEQ, MAX_WORD)
test_ds  = NERDataset(test_samples,  word_vocab, char_vocab, tag_map, MAX_SEQ, MAX_WORD)

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  collate_fn=collate_batch, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, collate_fn=collate_batch, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, collate_fn=collate_batch)

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}")

## 6. Fine-tune

In [ ]:
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from training.evaluate import run_evaluation

LR          = 5e-5
MAX_EPOCHS  = 25
PATIENCE    = 6
GRAD_CLIP   = 5.0
USE_FP16    = device.type == "cuda"
CKPT_DIR    = Path("/kaggle/working/checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4,
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=3, factor=0.5)
scaler    = GradScaler() if USE_FP16 else None

print(f"Optimizer: AdamW  |  lr={LR}  |  fp16={USE_FP16}")
print(f"Max epochs: {MAX_EPOCHS}  |  Early stop patience: {PATIENCE}")

In [ ]:
history       = []
best_f1       = 0.0
patience_cnt  = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    total_loss = 0
    n_batches  = 0
    t0         = time.time()

    for batch in train_loader:
        word_ids = batch["word_ids"].to(device)
        char_ids = batch["char_ids"].to(device)
        tag_ids  = batch["tag_ids"].to(device)
        lang_ids = batch["lang_ids"].to(device)
        mask     = batch["mask"].to(device)
        lengths  = batch["lengths"].to(device)

        optimizer.zero_grad()

        if USE_FP16:
            with autocast():
                loss = model(word_ids, char_ids, lang_ids, tag_ids, mask, lengths)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = model(word_ids, char_ids, lang_ids, tag_ids, mask, lengths)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    avg_loss = total_loss / n_batches
    elapsed  = time.time() - t0

    val_metrics = run_evaluation(model, val_loader, tag_map, device)
    val_f1      = val_metrics["overall"]["f1"]
    val_prec    = val_metrics["overall"]["precision"]
    val_rec     = val_metrics["overall"]["recall"]

    scheduler.step(val_f1)
    lr_now = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch":   epoch + 1,
        "loss":    round(avg_loss, 4),
        "val_f1":  round(val_f1,  4),
        "val_p":   round(val_prec, 4),
        "val_r":   round(val_rec,  4),
        "lr":      lr_now,
    })

    marker = ""
    if val_f1 > best_f1:
        best_f1      = val_f1
        patience_cnt = 0
        torch.save({"epoch": epoch, "best_f1": best_f1, "model": model.state_dict()},
                   CKPT_DIR / "domain_finetuned_best.pt")
        marker = " ✓ best"
    else:
        patience_cnt += 1

    print(f"Epoch {epoch+1:02d}/{MAX_EPOCHS} | loss: {avg_loss:.4f} | "
          f"val_f1: {val_f1:.4f} | P: {val_prec:.4f} R: {val_rec:.4f} | "
          f"lr: {lr_now:.2e} | {elapsed:.1f}s{marker}")

    if patience_cnt >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

print(f"\nDone. Best val F1: {best_f1:.4f}")

## 7. Training Curve

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

epochs  = [h["epoch"]   for h in history]
losses  = [h["loss"]    for h in history]
f1s     = [h["val_f1"]  for h in history]
precs   = [h["val_p"]   for h in history]
recs    = [h["val_r"]   for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Operator Domain Fine-tuning", fontsize=14, fontweight="bold")

ax1.plot(epochs, losses, "o-", color="#e74c3c", linewidth=2, markersize=5, label="Train Loss")
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs, f1s,   "o-",  color="#2ecc71", linewidth=2, markersize=5, label="F1")
ax2.plot(epochs, precs, "s--", color="#3498db", linewidth=1.5, markersize=4, label="Precision")
ax2.plot(epochs, recs,  "^--", color="#e67e22", linewidth=1.5, markersize=4, label="Recall")
ax2.axhline(y=best_f1, color="gray", linestyle=":", alpha=0.7, label=f"Best F1={best_f1:.3f}")
ax2.set_title("Validation Metrics")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1.05)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/domain_finetune_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /kaggle/working/domain_finetune_curves.png")

## 8. Final Test Set Evaluation

In [ ]:
best_ckpt = torch.load(CKPT_DIR / "domain_finetuned_best.pt", map_location=device, weights_only=False)
model.load_state_dict(best_ckpt["model"])

test_metrics = run_evaluation(model, test_loader, tag_map, device)

print("=" * 50)
print("DOMAIN FINE-TUNED MODEL — TEST SET")
print("=" * 50)
print(f"  Overall F1        : {test_metrics['overall']['f1']:.4f}")
print(f"  Overall Precision : {test_metrics['overall']['precision']:.4f}")
print(f"  Overall Recall    : {test_metrics['overall']['recall']:.4f}")
print()
print(f"  {'Entity':<10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
print("  " + "-" * 46)
for ent, vals in sorted(test_metrics["per_entity"].items()):
    print(f"  {ent:<10} {vals['precision']:>10.3f} {vals['recall']:>10.3f} {vals['f1-score']:>10.3f} {vals['support']:>10}")

## 9. Live Inference Test

In [ ]:
import re

LANG_MAP    = {"en": 0, "ru": 1}
CYRILLIC_RE = re.compile(r"[\u0400-\u04FF]")

def detect_lang(text):
    return "ru" if len(CYRILLIC_RE.findall(text)) > len(text.split()) * 0.3 else "en"

def predict(text):
    tokens  = text.strip().split()
    lang    = detect_lang(text)
    lang_id = LANG_MAP.get(lang, 0)
    mwl     = cfg["data"]["max_word_len"]

    lowered  = [t.lower() for t in tokens]
    word_ids = torch.tensor([[word_vocab.encode(t) for t in lowered]], dtype=torch.long).to(device)
    char_ids = torch.tensor([[char_vocab.encode_word(t, mwl) for t in lowered]], dtype=torch.long).to(device)
    lang_ids = torch.tensor([lang_id], dtype=torch.long).to(device)
    mask     = torch.ones(1, len(tokens), dtype=torch.float).to(device)
    lengths  = torch.tensor([len(tokens)], dtype=torch.long).to(device)

    model.eval()
    with torch.no_grad():
        pred_ids = model.predict(word_ids, char_ids, lang_ids, mask, lengths)

    tags = [tag_map.decode(pred_ids[0][i].item()) for i in range(len(tokens))]
    return list(zip(tokens, tags))

def extract_entities(text):
    tagged   = predict(text)
    entities = []
    cur_type, cur_toks = None, []
    for tok, tag in tagged:
        if tag.startswith("B-"):
            if cur_type: entities.append((cur_type, " ".join(cur_toks)))
            cur_type, cur_toks = tag[2:], [tok]
        elif tag.startswith("I-") and cur_type == tag[2:]:
            cur_toks.append(tok)
        else:
            if cur_type: entities.append((cur_type, " ".join(cur_toks)))
            cur_type, cur_toks = None, []
    if cur_type: entities.append((cur_type, " ".join(cur_toks)))
    return entities

test_sentences = [
    "I want to activate the Unlimited plan by dialing *100#",
    "Please enable mobile internet on my account",
    "Я хочу подключить тариф Premium отправив СТАРТ на *111#",
    "Подключите мне мобильный интернет пожалуйста",
    "John Smith from MegaFon asked about the Gold tariff in Moscow",
    "Send BALANCE to check your account status",
    "Сергей Петров из МТС хочет отключить голосовую почту",
    "Dial *555# to subscribe to the Platinum package",
    "How much does the Family plan cost per month",
    "Отправьте БАЛАНС для проверки счёта",
]

print("=" * 65)
print("ENTITY EXTRACTION — DOMAIN FINE-TUNED MODEL")
print("=" * 65)

for text in test_sentences:
    entities = extract_entities(text)
    lang     = detect_lang(text)
    print(f"\n[{lang.upper()}] {text}")
    if entities:
        for etype, etext in entities:
            print(f"       → [{etype:4s}] {etext}")
    else:
        print("       → (no entities)")

## 10. Token-level Visualization

In [ ]:
DEMO_TEXT = "Activate the Unlimited tariff by dialing *100# or send BALANCE to check status"

tagged = predict(DEMO_TEXT)

COLOR = {
    "B-MISC": "\033[95m", "I-MISC": "\033[95m",
    "B-PER":  "\033[94m", "I-PER":  "\033[94m",
    "B-ORG":  "\033[93m", "I-ORG":  "\033[93m",
    "B-LOC":  "\033[92m", "I-LOC":  "\033[92m",
    "O":      "\033[0m",
}
RESET = "\033[0m"

print("Token-level tagging:")
print(f"  {'Token':<22} {'Tag'}")
print("  " + "-" * 32)
for token, tag in tagged:
    color = COLOR.get(tag, "")
    print(f"  {color}{token:<22} {tag}{RESET}")

## 11. Save Results

In [ ]:
results = {
    "domain_finetune": {
        "best_val_f1":       round(best_f1, 4),
        "test_f1":           round(test_metrics["overall"]["f1"], 4),
        "test_precision":    round(test_metrics["overall"]["precision"], 4),
        "test_recall":       round(test_metrics["overall"]["recall"], 4),
        "per_entity":        test_metrics["per_entity"],
        "base_checkpoint":   "merged_best.pt",
        "freeze_word_emb":   FREEZE_WORD_EMB,
        "lr":                LR,
        "epochs_trained":    len(history),
        "history":           history,
    }
}

with open("/kaggle/working/domain_finetune_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Output files:")
for fp in Path("/kaggle/working").glob("**/*"):
    if fp.is_file() and fp.suffix in (".pt", ".json", ".png"):
        size = fp.stat().st_size / 1024
        print(f"  {str(fp.relative_to('/kaggle/working')):<50} {size:>8.1f} KB")